# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fin1806/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

The daily table has one row per content item, client, and report date. The sample covers 1 to 30 June 2026. It contains 6,390 repeated daily keys, and the extra rows are exact copies. After those copies are removed, the daily key is unique in the sample. For scoring, the daily rows become one row per content item and client at a chosen cutoff date. The one-month sample does not show how much history each client has in the full release.

In [8]:
import os
import getpass
import duckdb

# Use a read token from the environment or a hidden prompt; never save it in the notebook.
hf_token = os.environ.get('HF_TOKEN') or getpass.getpass('Hugging Face READ token: ')
con = duckdb.connect()
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '"
            + hf_token.replace("'", "''") + "')")
del hf_token

daily_sample = (
    "read_parquet('hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance_sample.parquet')"
)

row_count, client_count, first_day, last_day = con.sql(f"""
    SELECT COUNT(*), COUNT(DISTINCT client_hash_id),
           MIN(report_date), MAX(report_date)
    FROM {daily_sample}
""").fetchone()
print(f'Sample: {row_count:,} daily rows across {client_count} clients')
print(f'Observed sample dates: {first_day} to {last_day}')

duplicate_key_groups, extra_rows, null_key_groups = con.sql(f"""
    WITH keys AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM {daily_sample}
        GROUP BY 1, 2, 3
    )
    SELECT SUM(CASE WHEN n > 1 THEN 1 ELSE 0 END),
           SUM(n - 1),
           SUM(CASE WHEN n > 1 AND
                    (report_date IS NULL OR client_hash_id IS NULL OR content_hash_id IS NULL)
                    THEN 1 ELSE 0 END)
    FROM keys
""").fetchone()

distinct_rows = con.sql(f"""
    SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {daily_sample})
""").fetchone()[0]
exact_duplicate_rows = row_count - distinct_rows
print(f'Repeated daily keys: {duplicate_key_groups:,}; extra rows: {extra_rows:,}')
print(f'Exact duplicate rows: {exact_duplicate_rows:,}')

assert null_key_groups == 0, 'Repeated keys contain missing identifiers or dates'
assert extra_rows == exact_duplicate_rows, 'Some same-key rows differ; inspect before deduplicating'
daily_sample_unique = f'(SELECT DISTINCT * FROM {daily_sample})'
print('Daily key is unique after removing exact duplicate copies')


Sample: 11,694,072 daily rows across 65 clients
Observed sample dates: 2026-06-01 to 2026-06-30
Repeated daily keys: 6,390; extra rows: 6,390
Exact duplicate rows: 6,390
Daily key is unique after removing exact duplicate copies


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

The scoring unit is one content item for one client at a cutoff date. The feature window is the 30 days ending on that date. The following 30 days form the label window. The June sample can check the source fields, but it is too short to calculate both windows.

**Features:** prior30_impressions, prior30_clicks, and prior30_avg_position come from gsc_impressions, gsc_clicks, and gsc_avg_position in the feature window. A missing position stays missing; it is not a measured position.

**Label:** The proposed future_decline_label is 1 when impressions in the following 30 days (future30_impressions) are more than 20% lower than prior30_impressions. A page qualifies only if both 30-day windows are observed and the first window has at least 100 impressions. This label still needs to be calculated and tested on the full daily history. It measures a later search decline, not the effect of a content refresh.

**Context:** report_date, client_hash_id, and content_hash_id identify the date, client, and content item. gsc_data_start, ga4_data_start, and ga4_data_available help check which history is usable. Client IDs are also used to keep the same client together in a train/test split. These fields are not model features.

**Excluded:** future30_impressions and future_decline_label cannot be inputs because they contain the answer. The query table's 90-day metrics are excluded until their dates are checked against the cutoff. GA4 values are excluded when ga4_data_available is false because unavailable values may be filled with zero.

The output is a list of pages ranked for human review. The starter CSV fields is_declining_label and trend_direction are not used in this warehouse contract.

In [ ]:
# Verify that the proposed contract names fields in the warehouse daily schema.
daily_columns = {row[0] for row in con.sql(f"DESCRIBE SELECT * FROM {daily_sample}").fetchall()}
required_daily = {
    'report_date', 'client_hash_id', 'content_hash_id',
    'gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
    'ga4_data_available',
}
missing_daily = required_daily - daily_columns
assert not missing_daily, f'Missing warehouse fields: {sorted(missing_daily)}'
assert 'is_declining_label' not in daily_columns, 'Recheck the source: a label already exists'
print('Verified daily source fields:', ', '.join(sorted(required_daily)))
print('No built-in decline label in the daily source; the future-window label is proposed')

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The June sample has 11,694,072 daily rows across 65 clients. Removing the 6,390 exact copies leaves 11,687,682 rows. These are daily observations, not one row per page. The query below reports the number of client-content pairs, the date range, and missing key or search values. The sample's one-month span cannot provide both 30-day windows needed for the proposed label.

In [ ]:
# Aggregate-only audit of the exact-copy-deduplicated daily sample.
(rows, content_items, clients, first_observed, last_observed,
 missing_keys, missing_impressions, missing_clicks, missing_position) = con.sql(f"""
    SELECT COUNT(*), COUNT(DISTINCT (client_hash_id, content_hash_id)),
           COUNT(DISTINCT client_hash_id), MIN(report_date), MAX(report_date),
           COUNT(*) FILTER (WHERE report_date IS NULL OR client_hash_id IS NULL
                            OR content_hash_id IS NULL),
           COUNT(*) FILTER (WHERE gsc_impressions IS NULL),
           COUNT(*) FILTER (WHERE gsc_clicks IS NULL),
           COUNT(*) FILTER (WHERE gsc_avg_position IS NULL)
    FROM {daily_sample_unique}
""").fetchone()
print(f'Deduplicated daily rows: {rows:,}; distinct client-content pairs: {content_items:,}')
print(f'Clients: {clients}; observed dates: {first_observed} to {last_observed}')
print(f'Missing keys: {missing_keys:,}; missing impressions: {missing_impressions:,}; '
      f'missing clicks: {missing_clicks:,}; missing position: {missing_position:,}')
assert rows == distinct_rows, 'Deduplicated row count changed'
assert missing_keys == 0, 'Some daily observations lack a date or identifier'
print(f'Calendar span: {(last_observed - first_observed).days + 1} days')

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The June sample has 6,390 duplicate rows that are exact copies. They are removed before the sample checks. The full release has not been checked for the same issue.

Client history is uneven because clients started collecting data on different dates. A page needs a complete 30-day feature window and 30-day label window before it can be used for the proposed label. The June sample cannot confirm that coverage.

Some GA4 values are filled with zero when ga4_data_available is false. Those zeros mean the data was unavailable, not that there was no engagement. The query below counts how often this flag occurs in the June sample. The proposed features use GSC values.

The query table covers a fixed 90-day period. Its metrics may include dates after the cutoff, so they are excluded until the windows are aligned.

The data can support a directional ranking of pages for review. It cannot show that a content refresh caused traffic to change.

In [ ]:
# Aggregate checks only: no client identifiers or private rows are displayed.
dim_clients = (
    "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"
)
(client_rows, first_gsc_start, last_gsc_start,
 missing_gsc_start, first_ga4_start, last_ga4_start,
 missing_ga4_start) = con.sql(f"""
    SELECT COUNT(*), MIN(gsc_data_start), MAX(gsc_data_start),
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL),
           MIN(ga4_data_start), MAX(ga4_data_start),
           COUNT(*) FILTER (WHERE ga4_data_start IS NULL)
    FROM {dim_clients}
""").fetchone()
(ga4_unavailable, ga4_flag_missing) = con.sql(f"""
    SELECT COUNT(*) FILTER (WHERE ga4_data_available = FALSE),
           COUNT(*) FILTER (WHERE ga4_data_available IS NULL)
    FROM {daily_sample_unique}
""").fetchone()
print(f'Client dimension rows: {client_rows:,}')
print(f'GSC start range: {first_gsc_start} to {last_gsc_start}; missing: {missing_gsc_start:,}')
print(f'GA4 start range: {first_ga4_start} to {last_ga4_start}; missing: {missing_ga4_start:,}')
print(f'Deduplicated June rows with GA4 unavailable: {ga4_unavailable:,}; '
      f'flag missing: {ga4_flag_missing:,}')

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.